In [6]:
import os
from pathlib import Path

# Walk up from the notebook's location until we find the project root
# (identified by the presence of the 'src' directory)
project_root = Path.cwd()
while not (project_root / 'src').exists() and project_root != project_root.parent:
    project_root = project_root.parent
os.chdir(project_root)
print(os.getcwd())


C:\Users\Klara\retail-intelligence


In [7]:
import pandas as pd

alerts = pd.read_csv('data/processed/inventory_alerts.csv')

print("Alert distribution:")
print(alerts['AlertStatus'].value_counts())

Alert distribution:
AlertStatus
🔴 Stockout Risk          11
🟢 Adequate                6
⚪ Unreliable Forecast     2
🟡 Reorder Soon            1
Name: count, dtype: int64


In [8]:
print("Critical alerts (need immediate action):")
critical = alerts[alerts['AlertLevel'] == 'critical']
print(critical[['StockCode', 'Description', 'EstimatedCurrentStock',
                 'ForecastedDemand_Reorder', 'SuggestedReorderQty',
                 'AlertMessage']].to_string())

Critical alerts (need immediate action):
   StockCode                        Description  EstimatedCurrentStock  ForecastedDemand_Reorder  SuggestedReorderQty                                                            AlertMessage
0      21915             RED  HARMONICA IN BOX              937.038274               1538.915105               1002.0   Stock (937) below reorder point (1419). Order 1002 units immediately.
1      22197                     POPCORN HOLDER            2538.434359               5011.338545               3494.0  Stock (2538) below reorder point (4333). Order 3494 units immediately.
2      22178    VICTORIAN GLASS HANGING T-LIGHT            1217.653451               1542.320757                587.0   Stock (1218) below reorder point (1287). Order 587 units immediately.
3      22086    PAPER CHAIN KIT 50'S CHRISTMAS              871.948047               2241.445534               1959.0   Stock (872) below reorder point (2067). Order 1959 units immediately.
4      23

In [10]:
print("Warning alerts (reorder soon):")
warning = alerts[alerts['AlertLevel'] == 'warning']
print(warning[['StockCode', 'Description', 'EstimatedCurrentStock',
                 'ForecastedDemand_Reorder', 'SuggestedReorderQty',
                 'AlertMessage']].to_string())

Warning alerts (reorder soon):
   StockCode                         Description  EstimatedCurrentStock  ForecastedDemand_Reorder  SuggestedReorderQty                                                                AlertMessage
11     21977  PACK OF 60 PINK PAISLEY CAKE CASES            1107.089996               1177.857017                322.0  Stock (1107) below 3-week forecast (1178). Order 322 units within 2 weeks.


In [11]:
import plotly.express as px

status_counts = alerts['AlertStatus'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']

fig1 = px.pie(
    status_counts, names='Status', values='Count',
    title='Inventory Alert Status Distribution',
    color='Status',
    color_discrete_map={
        '🔴 Stockout Risk': '#d62728',
        '🟡 Reorder Soon': '#ff7f0e',
        '🟣 Overstock': '#9467bd',
        '🟢 Adequate': '#2ca02c',
        '⚪ Unreliable Forecast': '#7f7f7f'
    },
    hole=0.4
)
fig1.show()

In [13]:
import plotly.graph_objects as go

ok_alerts = alerts[alerts['AlertLevel'] != 'unreliable'].copy()

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    name='Estimated Current Stock', x=ok_alerts['StockCode'],
    y=ok_alerts['EstimatedCurrentStock'], marker_color='#1f77b4'
))
fig2.add_trace(go.Bar(
    name='Forecasted Demand (3wk)', x=ok_alerts['StockCode'],
    y=ok_alerts['ForecastedDemand_Reorder'], marker_color='#ff7f0e'
))
fig2.add_trace(go.Scatter(
    name='Reorder Point', x=ok_alerts['StockCode'],
    y=ok_alerts['ReorderPoint'], mode='markers',
    marker=dict(symbol='line-ew', size=20, color='red', line=dict(width=3, color='red'))
))
fig2.update_layout(
    barmode='group',
    title='Current Stock vs Forecasted Demand vs Reorder Point (OK products only)',
    xaxis_title='Product', yaxis_title='Units', height=500
)
fig2.show()

In [14]:
reorder_needed = ok_alerts[ok_alerts['SuggestedReorderQty'] > 0].copy()

fig3 = px.bar(
    reorder_needed.sort_values('SuggestedReorderQty', ascending=False),
    x='StockCode', y='SuggestedReorderQty',
    title='Suggested Reorder Quantities by Product',
    color='AlertStatus',
    color_discrete_map={'🔴 Stockout Risk': '#d62728', '🟡 Reorder Soon': '#ff7f0e'},
    labels={'SuggestedReorderQty': 'Units to Reorder'}
)
fig3.update_layout(height=400)
fig3.show()

In [18]:
import pandas as pd

alerts = pd.read_csv('data/processed/inventory_alerts.csv')

stock_code = "21915"
product = alerts[alerts['StockCode'] == stock_code].iloc[0]


print(f"Product: {product['StockCode']} — {product['Description']}")
print(f"Status: {product['AlertStatus']}")
print()
print(f"Current stock:            {product['EstimatedCurrentStock']:.0f} units")
print(f"Lead-time forecast:       {product['ForecastedDemand_Lead']:.0f} units")
print(f"3-week forecast:          {product['ForecastedDemand_Reorder']:.0f} units")
print(f"Safety stock:             {product['SafetyStock']:.0f} units")
print(f"Reorder point:            {product['ReorderPoint']:.0f} units")
print(f"Suggested reorder qty:    {product['SuggestedReorderQty']:.0f} units")
print()
print(f"Action: {product['AlertMessage']}")

Product: 21915 — RED  HARMONICA IN BOX 
Status: 🔴 Stockout Risk

Current stock:            937 units
Lead-time forecast:       1019 units
3-week forecast:          1539 units
Safety stock:             400 units
Reorder point:            1419 units
Suggested reorder qty:    1002 units

Action: Stock (937) below reorder point (1419). Order 1002 units immediately.


# Week5/Day3: Inventory Alert System

## Alert Distribution

- Out of 20 products, **11 (55%)** were flagged as **🔴 Stockout Risk**, **6 (30%)** as **🟢 Adequate**, **1 (5%)** as **🟡 Reorder Soon**, and **2 (10%)** as **⚪ Unreliable Forecast**.

- The two unreliable products (**23843** and **23166**) were excluded from the normal alert logic, confirming that the Week 4 data-quality flag propagated correctly into the inventory pipeline.

- The large number of Stockout Risk alerts is consistent with Week 4's finding that many products exhibit upward trends. Because current inventory is estimated using a backward-looking proxy (three weeks of historical average demand), the estimate naturally lags behind growing demand.

---

## Stock vs Demand Analysis

- For every product labeled **🔴 Stockout Risk**, the reorder point is higher than the estimated current stock, confirming that the alert logic behaves as intended.

- Product **85099F** has the narrowest margin between current stock (**879 units**) and reorder point (**904 units**), making its alert particularly sensitive to the synthetic stock assumption.

- Product **22197** shows the largest gap between estimated stock and forecasted demand, consistent with its Day 2 finding as the least predictable OK product (highest mean MAE).

- No contradictions were found between the visualizations and the computed alert statuses.

---

## Suggested Reorder Analysis

- Product **22197** receives the largest suggested reorder quantity (**3494 units**), reflecting both its high demand and its high forecasting uncertainty.

- Products **17003**, **22616**, **22469**, and **15036** are labeled **🟢 Adequate** despite having small positive reorder quantities.

- This apparent contradiction is expected because **SuggestedReorderQty** and **AlertStatus** are calculated from different thresholds:

  - **AlertStatus** compares current stock with the reorder point.
  - **SuggestedReorderQty** estimates how much inventory would fully cover future demand plus safety stock.

- Therefore, a product may be adequately stocked today while still benefiting from a small future reorder.

---

## Reorder Logic: Why Reorder Point ≠ Suggested Reorder Qty

Two different demand horizons drive these numbers, not one — this is what makes them look inconsistent at first glance:

- **Reorder Point** = `ForecastDemand_Lead + Safety Stock` — uses the *short* lead-time forecast (how much you'd sell while waiting for a new order to arrive). Answers: *"At what stock level am I at risk of running out? "* It's a **trigger**, not a quantity to buy.

- **Suggested Reorder Qty** = `ForecastDemand_Reorder + Safety Stock − EstimatedStock` — uses the *longer* 3-week reorder-cycle forecast (how much you'll sell before the next planned reorder review). Answers: *"How much should I actually buy so I don't have to reorder again almost immediately?"*

- **Safety Stock** = `z × forecast error std × √(lead time in weeks)`, where `z = 1.645` at the default 95% service level. Products with noisier historical forecasts (from Week 5's walk-forward validation) automatically get bigger buffers, since forecast uncertainty compounds with the square root of time rather than linearly.

**Business sense:** a retailer doesn't want to be told "buy exactly enough to reach your alarm threshold" — that would mean placing a new order again almost immediately. Instead, once triggered, the system recommends ordering enough to comfortably last through the *next* full reorder cycle, adjusted for how much this particular product's forecasts tend to miss by.


### Worked example: Product 21915 (RED HARMONICA IN BOX)

Running this logic on a real product from `inventory_alerts.csv` to confirm the formulas above produce self-consistent numbers:

- **Current stock:** 937 units (synthetic proxy — see Week 4/5 limitations note)
- **Lead-time forecast:** 1019 units → feeds only into the Reorder Point
- **3-week forecast:** 1539 units → feeds only into the Suggested Reorder Qty
- **Safety stock:** 400 units, scaled to this product's own forecast error
- **Reorder Point (1419):** 1019 + 400 — current stock (937) has already fallen below this, so the alert correctly fires as 🔴 Stockout Risk
- **Suggested Reorder Qty (1002):** 1539 + 400 − 937 — enough to cover the full 3-week cycle plus buffer, not just close the gap to the reorder point

---

## Business Interpretation

- The forecasting model predicts future demand, but the alert layer converts those predictions into concrete business actions.

- Instead of producing only numerical forecasts, the system generates:

  - 🔴 Stockout alerts
  - 🟡 Reorder recommendations
  - 🟣 Overstock warnings
  - 🟢 Adequate inventory confirmations

This business layer makes the forecasting pipeline actionable for inventory managers and demonstrates how machine-learning outputs can be translated into operational decisions.

---

## Limitations and Caveats

### 1. Synthetic inventory assumption

The dataset contains no real inventory information. Consequently, **EstimatedCurrentStock** is a synthetic proxy based on the historical three-week average demand.

All alerts and reorder quantities inherit this limitation and should be interpreted as illustrative rather than operational.

### 2. Forecast uncertainty

Suggested reorder quantities for highly volatile products, particularly **22197**, should be interpreted as directional guidance rather than precise recommendations.

Although these products pass the data-quality checks, their larger forecast errors increase uncertainty in the resulting inventory decisions.

### 3. Low-reliability products

Products **23843** and **23166** remain unsuitable for automated inventory decisions because of their sparse and spike-driven demand patterns.

The explicit **⚪ Unreliable Forecast** label prevents these products from generating misleading reorder recommendations.

---

## Overall Conclusion

The inventory alert system behaves consistently with all previous findings from Weeks 4 and 5. Data-quality flags propagate correctly, the alert logic matches the visualizations, and products with higher forecast uncertainty receive larger safety-stock buffers.

Most importantly, the system transforms raw forecasts into actionable inventory recommendations, turning the forecasting model into a business-oriented decision-support tool rather than a purely statistical exercise.